# 02 – Data Preprocessing
**AlchemiX – Launch26 Phase 2**

## Objectives
1. Remove duplicate rows (keeping `tick` to avoid false duplicates)
2. KNN-impute missing non-target numeric values
3. Drop rows with missing target values
4. Validate final schema
5. Save cleaned CSVs

## Forbidden in this notebook
- Feature engineering
- Encoding or scaling
- Model training
- Historical statistics
- Removing `tick` or `status` columns

In [1]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import matplotlib.pyplot as plt

from ml.utils import (
    TRAFFIC_RAW_FILE, TELEMETRY_RAW_FILE, INCIDENT_RAW_FILE,
    TRAFFIC_CLEAN_FILE, TELEMETRY_CLEAN_FILE, INCIDENT_CLEAN_FILE,
)
from ml.preprocessing import (
    clean_traffic, clean_telemetry, clean_incident,
    missing_value_report,
)

print('Imports OK ✓')
print(f'  Traffic  : {TRAFFIC_RAW_FILE}')
print(f'  Telemetry: {TELEMETRY_RAW_FILE}')
print(f'  Incident : {INCIDENT_RAW_FILE}')

Imports OK ✓
  Traffic  : E:\AlchemiX\datasets\raw\link_traffic_history.csv
  Telemetry: E:\AlchemiX\datasets\raw\link_telemetry.csv
  Incident : E:\AlchemiX\datasets\raw\link_incident_history.csv


## 1. Load Raw Datasets

In [2]:
traffic_raw   = pd.read_csv(TRAFFIC_RAW_FILE)
telemetry_raw = pd.read_csv(TELEMETRY_RAW_FILE)
incident_raw  = pd.read_csv(INCIDENT_RAW_FILE)

print('Raw shapes:')
print(f'  Traffic  : {traffic_raw.shape}')
print(f'  Telemetry: {telemetry_raw.shape}')
print(f'  Incident : {incident_raw.shape}')

Raw shapes:
  Traffic  : (6000, 6)
  Telemetry: (6000, 4)
  Incident : (6000, 4)


## 2. Clean Traffic Dataset

In [3]:
print('─' * 60)
print('TRAFFIC CLEANING')
print('─' * 60)
print('Before:')
print(f'  Rows : {len(traffic_raw)}')
print(f'  Missing observed_latency_ms: {traffic_raw["observed_latency_ms"].isna().sum()}')
print(f'  Missing load_units         : {traffic_raw["load_units"].isna().sum()}')
print(f'  Status counts:\n{traffic_raw["status"].value_counts().to_string()}')

traffic_clean = clean_traffic(traffic_raw)

print('\nAfter:')
print(f'  Rows : {len(traffic_clean)}')
missing_value_report(traffic_clean, 'traffic_clean')
display(traffic_clean.head(3))

03:17:39  INFO      ml.preprocessing  [traffic] Schema OK – 6000 rows, 6 columns
03:17:39  INFO      ml.preprocessing  [traffic] Duplicates removed: 0 (rows remaining: 6000)
03:17:39  INFO      ml.preprocessing  [traffic] KNN imputation: 203 → 0 missing values in ['load_units', 'load_ratio']
03:17:39  INFO      ml.preprocessing  [traffic] Rows removed due to missing target 'observed_latency_ms': 257 (rows remaining: 5743)
03:17:39  INFO      ml.preprocessing  [traffic] No missing values remaining.
03:17:39  INFO      ml.preprocessing  [traffic] Cleaning complete – 5743 rows
03:17:39  INFO      ml.preprocessing  [traffic_clean] No missing values remaining.


────────────────────────────────────────────────────────────
TRAFFIC CLEANING
────────────────────────────────────────────────────────────
Before:
  Rows : 6000
  Missing observed_latency_ms: 257
  Missing load_units         : 203
  Status counts:
status
ok           5997
saturated       3

After:
  Rows : 5743


,link_id,tick,load_units,load_ratio,status,observed_latency_ms
0,Aegis-Boreas,0,90.0,0.4328,ok,118635.583
1,Aegis-Boreas,1,136.4,0.6557,ok,283818.179
2,Aegis-Boreas,3,55.8,0.2681,ok,76938.079


## 3. Clean Telemetry Dataset

In [4]:
print('─' * 60)
print('TELEMETRY CLEANING')
print('─' * 60)
print('Before:')
print(f'  Rows : {len(telemetry_raw)}')
print(f'  Missing self_reported_latency_ms: {telemetry_raw["self_reported_latency_ms"].isna().sum()}')
print(f'  Missing measured_latency_ms     : {telemetry_raw["measured_latency_ms"].isna().sum()}')

telemetry_clean = clean_telemetry(telemetry_raw)

print('\nAfter:')
print(f'  Rows : {len(telemetry_clean)}')
missing_value_report(telemetry_clean, 'telemetry_clean')
display(telemetry_clean.head(3))

03:17:39  INFO      ml.preprocessing  [telemetry] Schema OK – 6000 rows, 4 columns
03:17:39  INFO      ml.preprocessing  [telemetry] Duplicates removed: 0 (rows remaining: 6000)
03:17:39  INFO      ml.preprocessing  [telemetry] KNN imputation: 256 → 0 missing values in ['self_reported_latency_ms']
03:17:39  INFO      ml.preprocessing  [telemetry] Rows removed due to missing target 'measured_latency_ms': 0 (rows remaining: 6000)
03:17:39  INFO      ml.preprocessing  [telemetry] No missing values remaining.
03:17:39  INFO      ml.preprocessing  [telemetry] Cleaning complete – 6000 rows
03:17:39  INFO      ml.preprocessing  [telemetry_clean] No missing values remaining.


────────────────────────────────────────────────────────────
TELEMETRY CLEANING
────────────────────────────────────────────────────────────
Before:
  Rows : 6000
  Missing self_reported_latency_ms: 256
  Missing measured_latency_ms     : 0

After:
  Rows : 6000


,link_id,tick,self_reported_latency_ms,measured_latency_ms
0,Aegis-Boreas,0,69804.794,70246.059
1,Aegis-Boreas,1,438998.039,461750.906
2,Aegis-Boreas,2,69595.273,71962.130


## 4. Clean Incident Dataset

In [5]:
print('─' * 60)
print('INCIDENT CLEANING')
print('─' * 60)
print('Before:')
print(f'  Rows : {len(incident_raw)}')
print(f'  Missing traffic_share: {incident_raw["traffic_share"].isna().sum()}')
print(f'  jammed_flag value counts:\n{incident_raw["jammed_flag"].value_counts().to_string()}')

incident_clean = clean_incident(incident_raw)

print('\nAfter:')
print(f'  Rows : {len(incident_clean)}')
missing_value_report(incident_clean, 'incident_clean')
display(incident_clean.head(3))

03:17:39  INFO      ml.preprocessing  [incident] Schema OK – 6000 rows, 4 columns
03:17:39  INFO      ml.preprocessing  [incident] Duplicates removed: 0 (rows remaining: 6000)


────────────────────────────────────────────────────────────
INCIDENT CLEANING
────────────────────────────────────────────────────────────
Before:
  Rows : 6000
  Missing traffic_share: 252
  jammed_flag value counts:
jammed_flag
False    5507
True      493


03:17:39  INFO      ml.preprocessing  [incident] KNN imputation: 252 → 0 missing values in ['traffic_share']
03:17:39  INFO      ml.preprocessing  [incident] Rows removed due to missing target 'jammed_flag': 0 (rows remaining: 6000)
03:17:39  INFO      ml.preprocessing  [incident] No missing values remaining.
03:17:39  INFO      ml.preprocessing  [incident] Cleaning complete – 6000 rows
03:17:39  INFO      ml.preprocessing  [incident_clean] No missing values remaining.



After:
  Rows : 6000


,link_id,tick,traffic_share,jammed_flag
0,Aegis-Boreas,0,0.03833,False
1,Aegis-Dawn,0,0.05373,False
2,Aegis-Elysium,0,0.06344,True


## 5. Validation

In [6]:
# ── Check tick is preserved ─────────────────────────────────────────────────
for name, df in [('traffic', traffic_clean), ('telemetry', telemetry_clean), ('incident', incident_clean)]:
    assert 'tick' in df.columns, f'{name}: tick column missing!'
    assert 'link_id' in df.columns, f'{name}: link_id column missing!'
    assert df['tick'].isna().sum() == 0, f'{name}: tick has null values!'
    print(f'{name}: tick OK (range {df["tick"].min()}–{df["tick"].max()})  rows={len(df)}')

# ── Check no target imputation ───────────────────────────────────────────────
assert traffic_clean['observed_latency_ms'].isna().sum() == 0,  'target has nulls!'
assert telemetry_clean['measured_latency_ms'].isna().sum() == 0, 'target has nulls!'
assert incident_clean['jammed_flag'].isna().sum() == 0, 'target has nulls!'

# ── Status column preserved ──────────────────────────────────────────────────
assert 'status' in traffic_clean.columns, 'status column removed!'

print('\n✓ All validation checks passed.')

traffic: tick OK (range 0–499)  rows=5743
telemetry: tick OK (range 0–499)  rows=6000
incident: tick OK (range 0–499)  rows=6000

✓ All validation checks passed.


## 6. Before / After Comparison

In [7]:
comparison = pd.DataFrame({
    'Dataset': ['Traffic', 'Telemetry', 'Incident'],
    'Raw Rows': [len(traffic_raw), len(telemetry_raw), len(incident_raw)],
    'Clean Rows': [len(traffic_clean), len(telemetry_clean), len(incident_clean)],
    'Removed': [
        len(traffic_raw) - len(traffic_clean),
        len(telemetry_raw) - len(telemetry_clean),
        len(incident_raw) - len(incident_clean),
    ],
})
comparison['Removed %'] = (comparison['Removed'] / comparison['Raw Rows'] * 100).round(2)
display(comparison)

,Dataset,Raw Rows,Clean Rows,Removed,Removed %
0,Traffic,6000,5743,257,4.28
1,Telemetry,6000,6000,0,0.00
2,Incident,6000,6000,0,0.00


## 7. Save Cleaned Datasets

In [8]:
traffic_clean.to_csv(TRAFFIC_CLEAN_FILE,   index=False)
telemetry_clean.to_csv(TELEMETRY_CLEAN_FILE, index=False)
incident_clean.to_csv(INCIDENT_CLEAN_FILE,  index=False)

print('Saved:')
print(f'  {TRAFFIC_CLEAN_FILE}')
print(f'  {TELEMETRY_CLEAN_FILE}')
print(f'  {INCIDENT_CLEAN_FILE}')
print('\n✓ Notebook 02 complete – proceed to Notebook 03 Feature Engineering.')

Saved:
  E:\AlchemiX\datasets\cleaned\traffic_clean.csv
  E:\AlchemiX\datasets\cleaned\telemetry_clean.csv
  E:\AlchemiX\datasets\cleaned\incident_clean.csv

✓ Notebook 02 complete – proceed to Notebook 03 Feature Engineering.
